## setup + imports

In [12]:
# === imports ===
import os, numpy as np, pandas as pd, torch, random
from tqdm.auto import tqdm
from utils.data_utils.load_data import load_dataset, extract_target_properties
from utils.evaluate_utils.load_paths import load_paths
from utils.evaluate_utils.load_models import load_config, load_decoder, load_fNN_model
from utils.evaluate_utils.sampling import *
from utils.evaluate_utils.structure_constraints import enforce_theta_domain, filter_S_candidates
from utils.evaluate_utils.error import compute_tensor_error
from utils.test_utils import *
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## load best parallel models and fNN

In [13]:
# === load best parallel models ===
MODELS = [
    ("001",18), 
    ("010",6), 
    ("100",9), 
    ("011",16), 
    ("101",5), 
    ("110",3), 
    ("111",5)
]

decoders   = []
configs    = []
P_means    = []
P_stds     = []
S_means    = []
S_stds     = []

for tag, trial in MODELS:
    paths = load_paths(
        TRIAL=trial,
        THETA_MODEL=True,
        THETA_PATTERN=tag,
        verbose=False
    )

    # load config + decoder
    config = load_config(paths["config_path"])
    decoder = load_decoder(config, paths["decoder_path"], flow_type=config.get("FLOW_TYPE","planar"),
                       trial=trial, device=device)

    # load per-model stats (fixed across trials for a given tag)
    Pm = np.load(paths["P_mean_path"]);  Ps = np.load(paths["P_std_path"])
    Sm = np.load(paths["S_mean_path"]);  Ss = np.load(paths["S_std_path"])
    Ps = np.where(Ps < 1e-8, 1.0, Ps)
    Ss = np.where(Ss < 1e-8, 1.0, Ss)

    # stash
    decoders.append(decoder)
    configs.append(config)
    P_means.append(Pm); P_stds.append(Ps)
    S_means.append(Sm); S_stds.append(Ss)

# === load max's forward model ===
fNN = load_fNN_model()

/Users/ellielin/Desktop/dresden/inverse_design_spinodoids/spinodoid_cvae_dev/utils/evaluate_utils/load_models.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  decoder.lo

✅ Loaded decoder from trial 18
✅ Loaded decoder from trial 6
✅ Loaded decoder from trial 9
✅ Loaded decoder from trial 16
✅ Loaded decoder from trial 5
✅ Loaded decoder from trial 3
✅ Loaded decoder from trial 5
✅ Loaded Max's forward model


## load test dataset + choose target sample

In [14]:
# === load test data ===
TEST_CSV = "data/test/dataset_test_x1000_augmented.csv"
P_all, S_all, C_all = load_dataset(TEST_CSV)
P_all = P_all.to(device)
S_all = S_all.to(device)
print(f"✅ Test set loaded: N={len(P_all)} | P: {tuple(P_all.shape)} | S: {tuple(S_all.shape)} | C: {C_all.shape}")

# === config for evaluation ===
PROB_THRESHOLD = 0.10
SAMPLES_PER_P  = 1000
PASS_THRESHOLD = 0.04
BW_MODE        = "auto"

# === pick test item ===
target_idx = 5
S_true = S_all[target_idx].detach().cpu().numpy().flatten()
P_true = P_all[target_idx].detach().cpu().numpy().flatten()
C_true = C_all[target_idx]

print(f"✅ Target index {target_idx}")
print(f"   - S_true: {np.array2string(S_true, formatter={'float_kind': lambda x: f'{x:.4f}'})}")
print(f"   - P_true: {np.array2string(P_true, formatter={'float_kind': lambda x: f'{x:.4f}'})}")

✅ Test set loaded: N=5001 | P: (5001, 9) | S: (5001, 4) | C: (5001, 3, 3, 3, 3)
✅ Target index 5
   - S_true: [87.3751 34.4427 0.0000 0.5033]
   - P_true: [0.2489 0.0823 0.0847 0.2574 0.0838 0.2660 0.0839 0.0863 0.0862]


## run target sample on all models

1. fNN baseline: fNN(S_true) = P_hat_fNN
2. for each theta model: 
        - theta(P_true) = S_hat_theta
        - fNN(S_hat_theta) = P_hat_theta
3. compare P_hat_fNN and P_hat_theta to P_true

In [15]:
# === baseline: fNN evaluated at S_true (table) ===
S_true_tf = np.expand_dims(S_true, axis=(0, 1))
C_fNN_true = fNN(S_true_tf).numpy().reshape(1,3,3,3,3)[0]

err_base = compute_tensor_error(C_true, C_fNN_true)   # C-tensor error (fraction)
base_status = "✅" if err_base < PASS_THRESHOLD else "❌ FAIL"

df_fnn = pd.DataFrame([{
    "Ŝ":         format_array(S_true),                 # using S_true as Ŝ_fNN reference
    "ΔS":         format_array(np.zeros_like(S_true)),  # S_true - S_true
    "error":      f"{err_base:.4%}",                    # percent format
    "status":     base_status,
}])

print(f"\n⚪ fNN baseline prediction: fNN(S_true) = P_hat_fNN")
display(df_fnn)


df_max, S_star, Phi_star, loss = run_max_inverse_design(C_true, S_true)


# === evaluate target sample on each parallel model ===
for midx, (tag, trial) in enumerate(MODELS):
    decoder    = decoders[midx]
    cfg        = configs[midx]
    P_mean     = P_means[midx]; P_std  = P_stds[midx]
    S_mean     = S_means[midx]; S_std  = S_stds[midx]
    latent_dim = int(cfg["LATENT_DIM"])

    # normalize P for this model
    P_norm = (P_true - P_mean) / (P_std + 1e-8)
    P_norm_t = torch.tensor(P_norm, dtype=torch.float32, device=device).unsqueeze(0)

    # sample Ŝ (normalized)
    S_hats_norm = get_S_hats(decoder, P_norm_t, latent_dim, num_samples=SAMPLES_PER_P, seed=SEED, device=device)

    # peaks + bandwidth
    if isinstance(BW_MODE, str) and BW_MODE.lower() == "auto":
        S_hat_peaks_norm, bw_used = extract_peaks_with_bandwidth(
            S_hats_norm, use_auto_bandwidth=True, target_range=(1, 10), verbose=False
        )
    else:
        bw_used = float(BW_MODE)
        S_hat_peaks_norm = get_S_hat_peaks(S_hats_norm, bandwidth=bw_used)

    # sort + probability filter
    S_hat_peaks_norm, probs, _ = sort_and_select_peaks_by_probability(
        S_hats_norm, S_hat_peaks_norm, bw_used, prob_threshold=PROB_THRESHOLD, verbose=False
    )

    # denorm + constraints
    S_hat_peaks_unnorm = S_hat_peaks_norm * S_std + S_mean
    S_hat_peaks_unnorm = enforce_theta_domain(S_hat_peaks_unnorm)
    S_hat_peaks_unnorm = filter_S_candidates(S_hat_peaks_unnorm)

    # forward each candidate → Ĉ and compute error
    rows = []
    for S_hat in S_hat_peaks_unnorm:
        dS = S_hat - S_true
        S_hat_tf = np.expand_dims(S_hat, axis=(0, 1))
        C_pred = fNN(S_hat_tf).numpy().reshape(1,3,3,3,3)[0]
        err  = float(compute_tensor_error(C_true, C_pred))
        stat = "✅" if err < PASS_THRESHOLD else "❌ FAIL"
        interesting = "✅" if (np.abs(S_hat[:3] - S_true[:3]) >= 5.0).any() else "❌"
        rows.append({
            "Ŝ":    format_array(S_hat),
            "ΔS":   format_array(dS),
            "error":    f"{err:.4%}",
            "status":   stat,
            "interesting": interesting
        })

    df = pd.DataFrame(rows)
    print(f"\n⚪ {tag} — {len(df)} candidates | bw_used={bw_used:.3f}")
    display(df)


⚪ fNN baseline prediction: fNN(S_true) = P_hat_fNN


,Ŝ,ΔS,error,status
0,"[87.37508, 34.44271, 0.00000, 0.50335]","[0.00000, 0.00000, 0.00000, 0.00000]",1.6911%,✅


ModuleNotFoundError: No module named 'src'

In [ ]:
# === random N-sample summary across all 7 models ===

# config
N               = 50
PROB_THRESHOLD  = PROB_THRESHOLD
BW_MODE_LOCAL   = BW_MODE
SAMPLES         = SAMPLES_PER_P 

# small formatter to keep S_true readable
def _fmt(arr, precision=4):
    arr = np.asarray(arr, dtype=float).flatten()
    return "[" + ", ".join(f"{v:.{precision}f}" for v in arr) + "]"

# choose indices
all_N = len(P_all)
idxs = random.sample(range(all_N), k=min(N, all_N))
print(f"✅ Random sample: {len(idxs)} indices  → {idxs}")
print(f"   PASS<thr={PASS_THRESHOLD}, prob_thr={PROB_THRESHOLD}, bw={BW_MODE_LOCAL}, samples={SAMPLES}")

rows = []
tags = [t for (t, _) in MODELS]

for i in tqdm(idxs, desc="Evaluating random targets", unit="target"):
    # ground truth for this target
    P_true = P_all[i].detach().cpu().numpy().flatten()
    S_true = S_all[i].detach().cpu().numpy().flatten()
    C_true = C_all[i]

    # per-model counts
    per_pass = {}
    per_interesting = {}

    total_pass = 0
    total_interesting = 0

    for midx, (tag, trial) in enumerate(MODELS):
        decoder    = decoders[midx]
        cfg        = configs[midx]
        P_mean     = P_means[midx]; P_std  = P_stds[midx]
        S_mean     = S_means[midx]; S_std  = S_stds[midx]
        latent_dim = int(cfg["LATENT_DIM"])

        # normalize P for this model
        P_norm = (P_true - P_mean) / (P_std + 1e-8)
        P_norm_t = torch.tensor(P_norm, dtype=torch.float32, device=device).unsqueeze(0)

        # sample Ŝ (normalized)
        S_hats_norm = get_S_hats(decoder, P_norm_t, latent_dim,
                                 num_samples=SAMPLES, seed=SEED, device=device)

        # peaks + bandwidth
        if isinstance(BW_MODE_LOCAL, str) and BW_MODE_LOCAL.lower() == "auto":
            S_hat_peaks_norm, bw_used = extract_peaks_with_bandwidth(
                S_hats_norm, use_auto_bandwidth=True, target_range=(1, 10), verbose=False
            )
        else:
            bw_used = float(BW_MODE_LOCAL)
            S_hat_peaks_norm = get_S_hat_peaks(S_hats_norm, bandwidth=bw_used)

        # sort + probability filter
        S_hat_peaks_norm, _, _ = sort_and_select_peaks_by_probability(
            S_hats_norm, S_hat_peaks_norm, bw_used, prob_threshold=PROB_THRESHOLD, verbose=False
        )

        # denorm + constraints
        S_hat_peaks = S_hat_peaks_norm * S_std + S_mean
        S_hat_peaks = enforce_theta_domain(S_hat_peaks)
        S_hat_peaks = filter_S_candidates(S_hat_peaks)

        # count passes + "interesting among passed" (|Δtheta| >= 5 on any of first 3 components)
        pass_count = 0
        interesting_count = 0
        for S_peak in S_hat_peaks:
            S_peak_tf = np.expand_dims(S_peak, axis=(0, 1))  # (1,1,4)
            C_pred = fNN(S_peak_tf).numpy().reshape(1,3,3,3,3)[0]
            if compute_tensor_error(C_true, C_pred) < PASS_THRESHOLD:
                pass_count += 1
                if (np.abs(S_peak[:3] - S_true[:3]) >= 5.0).any():
                    interesting_count += 1

        per_pass[tag] = pass_count
        per_interesting[tag] = interesting_count
        total_pass += pass_count
        total_interesting += interesting_count

    # assemble row
    row = {"index": i, "S_true": _fmt(S_true)}
    for tag in tags:
        row[f"num_pass_{tag}"] = per_pass.get(tag, 0)
    row["total_num_passing"]    = total_pass
    row["total_interesting"]    = total_interesting   # among passed
    rows.append(row)

df_rand_summary = pd.DataFrame(rows)
display(df_rand_summary)

✅ Random sample: 50 indices  → [839, 4467, 712, 4837, 3456, 260, 244, 767, 1791, 1905, 4139, 4931, 217, 4597, 1628, 4464, 3436, 1805, 3679, 4827, 2278, 53, 1307, 3462, 2787, 2276, 1273, 1763, 2757, 837, 759, 3112, 792, 2940, 2817, 4945, 2166, 355, 3763, 4392, 1022, 3100, 645, 4522, 2401, 2962, 4729, 1575, 569, 375]
   PASS<thr=0.04, prob_thr=0.1, bw=auto, samples=1000


Evaluating random targets:   0%|          | 0/50 [00:00<?, ?target/s]

,index,S_true,num_pass_001,num_pass_010,num_pass_100,num_pass_011,num_pass_101,num_pass_110,num_pass_111,total_num_passing,total_interesting
0,839,"[74.7555, 31.4210, 0.0000, 0.9785]",1,1,2,3,4,5,4,20,19
1,4467,"[22.5113, 0.0000, 0.0000, 0.4997]",0,0,1,0,0,0,0,1,0
2,712,"[42.5440, 0.0000, 77.7432, 0.6283]",1,0,0,0,6,0,1,8,7
3,4837,"[0.0000, 0.0000, 80.5373, 0.8750]",1,0,0,3,6,0,1,11,10
4,3456,"[29.8441, 18.2028, 62.3237, 0.9137]",1,0,0,3,4,0,1,9,9
5,260,"[0.0000, 41.0198, 81.1721, 0.8692]",1,1,0,5,2,2,1,12,12
6,244,"[15.9600, 0.0000, 30.0434, 0.4709]",0,0,0,0,0,0,0,0,0
7,767,"[81.5878, 16.6593, 0.0000, 0.7041]",0,0,1,0,3,5,1,10,10
8,1791,"[0.0000, 47.7745, 18.8022, 0.6937]",0,0,0,1,0,0,0,1,0
9,1905,"[0.0000, 59.7617, 29.5949, 0.9582]",1,1,0,1,0,5,1,9,8
